# 🧠 Single Agent Pipeline Project

## Problem Statement
Build a **Single-Agent Smart Assistant** that:
- Understands user queries
- Routes tasks based on intent
- Uses tools when required
- Returns structured JSON output

### The agent should handle:
- Math queries → Calculator Tool
- Keyword extraction → Keyword Tool
- General queries → Direct response

---
### 🛠️ What You Need to Implement
- Agent logic
- Conditional routing
- Tool integration
- Basic error handling

### 🚀 Bonus
- Improve routing
- Add logging
- Add more tools


In [1]:
# 🛠️ TOOL 1: Calculator

def calculator(expression: str) -> str:
    """Evaluate a mathematical expression."""
    try:
        return str(eval(expression))
    except Exception:
        return "Error in calculation"

In [2]:
# 🛠️ TOOL 2: Keyword Extractor

def extract_keywords(text: str) -> list:
    """Extract keywords from text."""
    try:
        words = text.split()
        keywords = list(set([w.lower() for w in words if len(w) > 4]))
        return keywords[:5]
    except Exception:
        return []

In [3]:
def count_stats(text: str) -> dict:
    """Return word and character counts for a piece of text."""
    try:
        words = text.split()
        return {
            "word_count": len(words),
            "char_count": len(text)
        }
    except Exception:
        return {"word_count": 0, "char_count": 0}


In [4]:
import logging

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s"
)
logger = logging.getLogger("agent")


## 🤖 Implement Agent Logic Below

👉 Use conditional routing:
- If query contains "calculate" → use calculator
- If query contains "keywords" → use keyword extractor
- Else → general response

In [5]:
# 🤖 AGENT FUNCTION (IMPLEMENTED)

import re

def agent(query: str) -> dict:
    """
    Routes a natural-language query to the right tool based on intent, and
    returns a structured JSON-style response.

    Routing rules (bonus: improved beyond simple substring matching):
    - Math intent    -> Calculator Tool (looks for "calculate" OR an actual
                         arithmetic expression, so "20 + 5" alone still works)
    - Keyword intent  -> Keyword Tool (looks for "keyword(s)" OR "extract")
    - Stats intent    -> Word/Character Counter (bonus tool)
    - Anything else   -> General response
    - Empty input / unexpected errors -> "error" type, never raises
    """
    try:
        if not query or not query.strip():
            logger.warning("Received empty query.")
            return {"type": "error", "result": "Empty query received."}

        query_lower = query.lower().strip()
        has_math_expression = bool(re.search(r'\d+\s*[\+\-\*/]\s*\d+', query))

        if "calculate" in query_lower or has_math_expression:
            match = re.search(r'[\d\.\s\+\-\*/\(\)]+', query)
            expression = match.group().strip() if match else ""
            if not expression:
                logger.error(f"No valid expression found in query: {query}")
                return {"type": "error", "result": "Could not find a valid expression to calculate."}
            result = calculator(expression)
            logger.info(f"Routed to calculator | query='{query}' | result={result}")
            return {"type": "calculation", "result": result}

        elif "keyword" in query_lower or "extract" in query_lower:
            result = extract_keywords(query)
            logger.info(f"Routed to keyword extractor | query='{query}' | result={result}")
            return {"type": "keywords", "result": result}

        elif "count" in query_lower or "stats" in query_lower:
            result = count_stats(query)
            logger.info(f"Routed to stats tool | query='{query}' | result={result}")
            return {"type": "stats", "result": result}

        else:
            logger.info(f"Routed to general response | query='{query}'")
            return {"type": "general", "result": f"You asked: '{query}'. I don't have a specific tool for this yet."}

    except Exception as e:
        logger.exception(f"Unexpected error handling query: {query}")
        return {"type": "error", "result": f"Something went wrong: {str(e)}"}


## 📦 Expected Output Format

```
{
  "type": "calculation / keywords / general / error",
  "result": ...
}
```

In [6]:
# 🧪 Test Cases

queries = [
    "Calculate 20 + 5",
    "Extract keywords from Artificial Intelligence is transforming industries",
    "What is machine learning?"
]

for q in queries:
    print("Query:", q)
    print("Response:", agent(q))
    print("-" * 50)

2026-08-07 14:14:15,435 | INFO | Routed to calculator | query='Calculate 20 + 5' | result=25
2026-08-07 14:14:15,436 | INFO | Routed to keyword extractor | query='Extract keywords from Artificial Intelligence is transforming industries' | result=['keywords', 'transforming', 'industries', 'intelligence', 'extract']
2026-08-07 14:14:15,436 | INFO | Routed to general response | query='What is machine learning?'


Query: Calculate 20 + 5
Response: {'type': 'calculation', 'result': '25'}
--------------------------------------------------
Query: Extract keywords from Artificial Intelligence is transforming industries
Response: {'type': 'keywords', 'result': ['keywords', 'transforming', 'industries', 'intelligence', 'extract']}
--------------------------------------------------
Query: What is machine learning?
Response: {'type': 'general', 'result': "You asked: 'What is machine learning?'. I don't have a specific tool for this yet."}
--------------------------------------------------


In [7]:
# 🎯 Interactive Mode

while True:
    user_input = input("Enter query (type 'exit' to stop): ")
    if user_input.lower() == "exit":
        break
    print("Response:", agent(user_input))

2026-08-07 14:15:00,971 | INFO | Routed to keyword extractor | query='Extract keywords from what is life if not suffering everyday' | result=['suffering', 'extract', 'keywords', 'everyday']


Response: {'type': 'keywords', 'result': ['suffering', 'extract', 'keywords', 'everyday']}
